# Topical-Chat Resource-Constrained LLM Adaptation

This notebook runs four systems in the same Kaggle notebook:

1. Base Qwen
2. LoRA-only Qwen
3. RAG-only Qwen
4. LoRA + RAG Qwen

It also generates automatic comparison tables, latency summaries, retrieval-judgment sheets, and human-evaluation sheets for the paper.

In [ ]:
# Cell 1 - Install packages
# Restart the Kaggle session after this cell if packages are updated.
%pip install -q -U transformers==4.46.3 peft==0.13.2 accelerate==0.34.2 datasets==2.21.0 sentence-transformers==3.2.1

In [ ]:
.0/.g# Cell 2 - Imports and configuration
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # avoid DataParallel scatter bugs on Kaggle T4 x2
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HOME"] = "/kaggle/working/hf_home_topical_lora_rag"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import csv
import gc
import heapq
import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, Features, Sequence, Value, concatenate_datasets
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("/kaggle/working/qwen25_topical_lora_rag")
ADAPTER_DIR = OUTPUT_DIR / "adapter"

# If your trained adapter is uploaded as a Kaggle dataset, set this path.
# Example: ADAPTER_HINT = "/kaggle/input/qwen25-topical-chat-lora/adapter"
ADAPTER_HINT = None
if ADAPTER_HINT is not None:
    ADAPTER_DIR = Path(ADAPTER_HINT)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set this if auto-detection misses your Kaggle dataset path.
# Example: DATASET_HINT = "/kaggle/input/topical-chat-rag-data/topical-chat-rag-data"
DATASET_HINT = None

# LoRA training controls.
RUN_LORA_TRAINING = True          # set False if ADAPTER_DIR already exists and you only want inference/eval
RESUME_FROM_CHECKPOINT = True

# For a quick debug run, use TRAIN_FRACTION=0.02 and N_EVAL_CONTEXTS=5.
# For the paper, use TRAIN_FRACTION=0.60 or 1.0 and N_EVAL_CONTEXTS=50 or 100.
TRAIN_FRACTION = 0.5
EVAL_FRACTION = 0.25
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = 400

MAX_LENGTH = 256
MAX_HISTORY_TURNS = 6
NUM_TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 5e-6
WARMUP_RATIO = 0.05
SAVE_STEPS = 250

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# RAG controls.
# If auto-detection misses your uploaded local RAG corpus, add explicit paths here.
# Example: RAG_CORPUS_HINTS = ["/kaggle/input/my-rag-corpus/rag_corpus.jsonl"]
RAG_CORPUS_HINTS = []
RETRIEVAL_TOP_K = 3
RETRIEVAL_EVAL_TOP_K = 5
ACTIVE_RAG_RETRIEVER = "bm25"     # "bm25", "dense", or "hybrid"
ENABLE_DENSE_RETRIEVAL = False    # set True for dense/hybrid retrieval, slower but useful for paper
DENSE_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MAX_RAG_DOCS = 10000
MIN_DOC_CHARS = 80
RAG_CHUNK_WORDS = 120
RAG_CHUNK_STRIDE = 40

# Generation/evaluation controls.
N_EVAL_CONTEXTS = 50              # set 50 or 100 for the paper
MAX_NEW_TOKENS = 96
MAX_INFER_TOKENS = 768

SYSTEM_PROMPT = "You are a helpful conversational assistant. Continue the conversation naturally and reply as the assistant."

assert torch.cuda.is_available(), "Enable GPU in Kaggle."
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print("Visible GPUs:", torch.cuda.device_count())
print("Output dir:", OUTPUT_DIR)

In [ ]:
# Cell 3 - Locate Topical-Chat dataset files

def find_conversations_dir(dataset_hint=None):
    if dataset_hint:
        p = Path(dataset_hint)
        if (p / "conversations" / "train.json").exists():
            return p / "conversations"
        if (p / "train.json").exists() and (p / "valid_freq.json").exists() and (p / "valid_rare.json").exists():
            return p

    matches = []
    for train_file in Path("/kaggle/input").rglob("train.json"):
        parent = train_file.parent
        if (parent / "valid_freq.json").exists() and (parent / "valid_rare.json").exists():
            matches.append(parent)

    if not matches:
        raise FileNotFoundError("Could not find train.json, valid_freq.json, and valid_rare.json under /kaggle/input")

    return sorted(matches, key=lambda p: len(str(p)))[0]

CONV_DIR = find_conversations_dir(DATASET_HINT)
DATA_ROOT = CONV_DIR.parent if CONV_DIR.name == "conversations" else CONV_DIR
TRAIN_PATH = CONV_DIR / "train.json"
VALID_FREQ_PATH = CONV_DIR / "valid_freq.json"
VALID_RARE_PATH = CONV_DIR / "valid_rare.json"

print("DATA_ROOT:", DATA_ROOT)
print("CONV_DIR:", CONV_DIR)
print("TRAIN_PATH:", TRAIN_PATH)
print("VALID_FREQ_PATH:", VALID_FREQ_PATH)
print("VALID_RARE_PATH:", VALID_RARE_PATH)

In [ ]:
# Cell 4 - Tokenizer, ChatML rendering, and Topical-Chat preprocessing

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"


def normalize_text(text):
    return " ".join(str(text).strip().split())


def load_conversations(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        return list(data.values())
    if isinstance(data, list):
        return data
    raise ValueError(f"Unsupported JSON format in {json_path}")


def render_chatml(history, add_generation_prompt=False, system_prompt=SYSTEM_PROMPT):
    parts = [f"<|im_start|>system\n{system_prompt}<|im_end|>\n"]
    for msg in history:
        parts.append(f"<|im_start|>{msg['role']}\n{normalize_text(msg['content'])}<|im_end|>\n")
    if add_generation_prompt:
        parts.append("<|im_start|>assistant\n")
    return "".join(parts)


def tokenize_training_example(history, target_text):
    prompt_text = render_chatml(history, add_generation_prompt=True)
    target_text = normalize_text(target_text) + "<|im_end|>\n"

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target_text, add_special_tokens=False)["input_ids"]

    if len(target_ids) < 2:
        return None

    total_len = len(prompt_ids) + len(target_ids)
    if total_len > MAX_LENGTH:
        overflow = total_len - MAX_LENGTH
        if overflow >= len(prompt_ids):
            return None
        prompt_ids = prompt_ids[overflow:]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "length": len(input_ids),
        "target_tokens": len(target_ids),
    }


def iter_tokenized_examples(json_paths, fraction=None, seed=SEED):
    rng = random.Random(seed)
    for json_path in json_paths:
        conversations = load_conversations(json_path)
        if fraction is not None and 0 < fraction < 1.0:
            rng.shuffle(conversations)
            conversations = conversations[:max(1, int(len(conversations) * fraction))]

        for conv in conversations:
            history = []
            for turn in conv.get("content", []):
                text = normalize_text(turn.get("message", ""))
                if not text:
                    continue
                role = "assistant" if turn.get("agent") == "agent_2" else "user"

                if role == "assistant":
                    example = tokenize_training_example(history, text)
                    if example is not None:
                        yield example

                history.append({"role": role, "content": text})
                history = history[-MAX_HISTORY_TURNS:]

In [ ]:
# Cell 5 - Build tokenized LoRA train/eval datasets
FEATURES = Features(
    {
        "input_ids": Sequence(Value("int32")),
        "attention_mask": Sequence(Value("int8")),
        "labels": Sequence(Value("int32")),
        "length": Value("int32"),
        "target_tokens": Value("int32"),
    }
)

train_dataset = Dataset.from_generator(
    iter_tokenized_examples,
    gen_kwargs={"json_paths": [str(TRAIN_PATH)], "fraction": TRAIN_FRACTION, "seed": SEED},
    features=FEATURES,
    cache_dir=os.environ["HF_HOME"],
)

eval_freq_dataset = Dataset.from_generator(
    iter_tokenized_examples,
    gen_kwargs={"json_paths": [str(VALID_FREQ_PATH)], "fraction": EVAL_FRACTION, "seed": SEED + 1},
    features=FEATURES,
    cache_dir=os.environ["HF_HOME"],
)

eval_rare_dataset = Dataset.from_generator(
    iter_tokenized_examples,
    gen_kwargs={"json_paths": [str(VALID_RARE_PATH)], "fraction": EVAL_FRACTION, "seed": SEED + 2},
    features=FEATURES,
    cache_dir=os.environ["HF_HOME"],
)

eval_dataset = concatenate_datasets([eval_freq_dataset, eval_rare_dataset])
train_dataset = train_dataset.shuffle(seed=SEED)
eval_dataset = eval_dataset.shuffle(seed=SEED)

if MAX_TRAIN_SAMPLES is not None:
    train_dataset = train_dataset.select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))
if MAX_EVAL_SAMPLES is not None:
    eval_dataset = eval_dataset.select(range(min(MAX_EVAL_SAMPLES, len(eval_dataset))))

print("train examples:", len(train_dataset))
print("eval examples:", len(eval_dataset))

for i in range(min(32, len(train_dataset))):
    assert train_dataset[i]["target_tokens"] > 0, f"Bad train sample {i}"
for i in range(min(32, len(eval_dataset))):
    assert eval_dataset[i]["target_tokens"] > 0, f"Bad eval sample {i}"

In [ ]:
# Cell 6 - Label sanity check
sample = train_dataset[0]
target_ids = [tok for tok, lbl in zip(sample["input_ids"], sample["labels"]) if lbl != -100]
prompt_ids = [tok for tok, lbl in zip(sample["input_ids"], sample["labels"]) if lbl == -100]

print("sample length:", sample["length"])
print("target token count:", sample["target_tokens"])
print("\nPROMPT TAIL:\n")
print(tokenizer.decode(prompt_ids[-200:], skip_special_tokens=False))
print("\nTARGET:\n")
print(tokenizer.decode(target_ids, skip_special_tokens=False))

In [ ]:
# Cell 7 - Collator and optional LoRA model initialization
class CausalLMCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids, attention_mask, labels = [], [], []
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

data_collator = CausalLMCollator(tokenizer)

if RUN_LORA_TRAINING:
    torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float32,          # stable path for Kaggle; fp16 caused NaN in this project
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )
    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id

    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)
    model.enable_input_require_grads()
    model.print_trainable_parameters()
    model = model.cuda()

    sanity_batch = data_collator([train_dataset[0]])
    sanity_batch = {k: v.cuda() for k, v in sanity_batch.items()}
    model.eval()
    with torch.no_grad():
        sanity_loss = model(**sanity_batch).loss
    print("sanity loss:", float(sanity_loss))
    assert torch.isfinite(sanity_loss), "Sanity loss is not finite. Stop and inspect dtype/labels."
    model.train()
else:
    print("RUN_LORA_TRAINING=False; skipping training model initialization.")
    assert ADAPTER_DIR.exists(), f"Adapter not found: {ADAPTER_DIR}"

In [ ]:
# Cell 8 - LoRA training and saving
class StableTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
        if not torch.isfinite(loss):
            raise RuntimeError("NaN/Inf loss detected.")
        return (loss, outputs) if return_outputs else loss

if RUN_LORA_TRAINING:
    updates_per_epoch = max(1, math.ceil(len(train_dataset) / (TRAIN_BATCH_SIZE * GRAD_ACCUM)))
    print("optimizer steps per epoch:", updates_per_epoch)

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        overwrite_output_dir=False,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        weight_decay=0.01,
        max_grad_norm=0.3,
        fp16=False,
        bf16=False,
        optim="adamw_torch",
        logging_steps=20,
        logging_first_step=True,
        eval_strategy="no",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
        gradient_checkpointing=True,
        group_by_length=False,
        prediction_loss_only=True,
        seed=SEED,
        save_safetensors=True,
    )

    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )

    try:
        trainer = StableTrainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        trainer = StableTrainer(tokenizer=tokenizer, **trainer_kwargs)

    trainer.model_accepts_loss_kwargs = False

    last_checkpoint = None
    if RESUME_FROM_CHECKPOINT:
        checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
        if checkpoints:
            last_checkpoint = str(checkpoints[-1])
            print("Resuming from:", last_checkpoint)

    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    print(train_result.metrics)

    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    print("saved adapter:", ADAPTER_DIR)
else:
    print("Skipping LoRA training. Using existing adapter:", ADAPTER_DIR)

In [ ]:
# Cell 9 - Optional LoRA validation loss after training
if RUN_LORA_TRAINING:
    eval_for_metrics = eval_dataset.select(range(min(100, len(eval_dataset))))
    trainer.model.eval()
    eval_metrics = trainer.evaluate(eval_dataset=eval_for_metrics)
    eval_loss = eval_metrics["eval_loss"]
    eval_metrics["perplexity"] = math.exp(eval_loss) if math.isfinite(eval_loss) and eval_loss < 20 else float("inf")
    print(eval_metrics)
else:
    print("Skipped LoRA validation because RUN_LORA_TRAINING=False.")

In [ ]:
# Cell 10 - Build RAG document corpus from uploaded Topical-Chat/RAG files
TEXT_KEYWORDS = {
    "text", "content", "passage", "document", "body", "summary", "article", "evidence",
    "fact", "facts", "fun_facts", "wiki", "wiki_text", "shortened_wiki_lead_section",
    "summarized_wiki_lead_section", "fs", "factual", "section", "lead_section",
}
TITLE_KEYWORDS = {"title", "entity", "entity_name", "wiki_title", "name"}
ID_KEYWORDS = {"id", "doc_id", "document_id", "wiki_id", "source_id"}


def rag_word_tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", str(text).lower())


def flatten_strings(obj):
    strings = []
    if isinstance(obj, str):
        strings.append(obj)
    elif isinstance(obj, list):
        for item in obj:
            strings.extend(flatten_strings(item))
    elif isinstance(obj, dict):
        for value in obj.values():
            strings.extend(flatten_strings(value))
    return strings


def chunk_text(text, chunk_words=RAG_CHUNK_WORDS, stride=RAG_CHUNK_STRIDE):
    text = normalize_text(text)
    words = text.split()
    if len(words) < 20:
        return []
    if len(words) <= chunk_words:
        return [text]
    chunks = []
    step = max(1, chunk_words - stride)
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_words])
        if len(chunk.split()) >= 20:
            chunks.append(chunk)
        if start + chunk_words >= len(words):
            break
    return chunks


def get_first_present(record, keys, default=""):
    lower_map = {str(k).lower(): v for k, v in record.items()} if isinstance(record, dict) else {}
    for key in keys:
        if key in lower_map and lower_map[key]:
            return str(lower_map[key])
    return default


def add_text_doc(docs, text, title, source, doc_id):
    if len(docs) >= MAX_RAG_DOCS:
        return
    text = normalize_text(text)
    if len(text) < MIN_DOC_CHARS:
        return
    for chunk_index, chunk in enumerate(chunk_text(text)):
        if len(docs) >= MAX_RAG_DOCS:
            break
        docs.append({
            "doc_id": f"{doc_id}::chunk{chunk_index}",
            "title": title,
            "source": str(source),
            "text": chunk,
        })


def collect_docs_from_obj(obj, source, docs, default_title=None):
    if len(docs) >= MAX_RAG_DOCS:
        return

    if isinstance(obj, dict):
        title = get_first_present(obj, TITLE_KEYWORDS, default=default_title or Path(str(source)).stem)
        doc_id = get_first_present(obj, ID_KEYWORDS, default=f"{Path(str(source)).stem}_{len(docs)}")

        for key, value in obj.items():
            key_l = str(key).lower()
            looks_like_text = (
                key_l in TEXT_KEYWORDS
                or "wiki" in key_l
                or "section" in key_l
                or "fact" in key_l
                or "summary" in key_l
                or "passage" in key_l
            )
            if looks_like_text:
                for text in flatten_strings(value):
                    add_text_doc(docs, text, title=title, source=source, doc_id=f"{doc_id}_{key_l}")

        for key, value in obj.items():
            collect_docs_from_obj(value, source, docs, default_title=title)

    elif isinstance(obj, list):
        for item in obj:
            collect_docs_from_obj(item, source, docs, default_title=default_title)

    elif isinstance(obj, str):
        add_text_doc(docs, obj, title=default_title or Path(str(source)).stem, source=source, doc_id=f"{Path(str(source)).stem}_{len(docs)}")


def load_docs_from_file(path):
    path = Path(path)
    docs = []
    suffix = path.suffix.lower()
    try:
        if suffix == ".jsonl":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    if len(docs) >= MAX_RAG_DOCS:
                        break
                    line = line.strip()
                    if line:
                        collect_docs_from_obj(json.loads(line), str(path), docs)
        elif suffix == ".json":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                collect_docs_from_obj(json.load(f), str(path), docs)
        elif suffix == ".csv":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for row in csv.DictReader(f):
                    if len(docs) >= MAX_RAG_DOCS:
                        break
                    collect_docs_from_obj(row, str(path), docs)
        elif suffix == ".txt":
            add_text_doc(docs, path.read_text(encoding="utf-8", errors="ignore"), path.stem, str(path), path.stem)
    except Exception as e:
        print("Could not load:", path, repr(e))
    return docs


def auto_find_rag_files():
    roots = [Path("/kaggle/input")]
    if DATA_ROOT.exists():
        roots.insert(0, DATA_ROOT)

    suffixes = {".json", ".jsonl", ".csv", ".txt"}
    path_keywords = ["wiki", "rag", "corpus", "document", "documents", "reading", "knowledge", "evidence", "fs", "fact"]
    conversation_names = {"train.json", "valid_freq.json", "valid_rare.json", "test_freq.json", "test_rare.json"}

    found = []
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in suffixes:
                continue
            if path.parent == CONV_DIR and path.name in conversation_names:
                continue
            path_text = str(path).lower()
            if any(k in path_text for k in path_keywords):
                found.append(path)
    return sorted(set(found), key=lambda p: str(p))


def build_fallback_conversation_corpus(json_paths):
    docs = []
    for json_path in json_paths:
        for conv_idx, conv in enumerate(load_conversations(json_path)):
            messages = [normalize_text(t.get("message", "")) for t in conv.get("content", []) if normalize_text(t.get("message", ""))]
            add_text_doc(docs, " ".join(messages), "Topical-Chat conversation fallback", str(json_path), f"{Path(json_path).stem}_{conv_idx}")
            if len(docs) >= MAX_RAG_DOCS:
                break
    return docs

rag_candidate_files = [Path(p) for p in RAG_CORPUS_HINTS] if RAG_CORPUS_HINTS else auto_find_rag_files()
print("candidate RAG files:", len(rag_candidate_files))
for p in rag_candidate_files[:20]:
    print(" -", p)

rag_docs = []
for path in rag_candidate_files:
    if len(rag_docs) >= MAX_RAG_DOCS:
        break
    rag_docs.extend(load_docs_from_file(path))
    rag_docs = rag_docs[:MAX_RAG_DOCS]

if not rag_docs:
    print("WARNING: No external wiki/RAG corpus found. Using conversation fallback only for code testing.")
    print("Do not report fallback conversation retrieval as factual RAG in the paper.")
    rag_docs = build_fallback_conversation_corpus([VALID_FREQ_PATH, VALID_RARE_PATH, TRAIN_PATH])

print("RAG docs/chunks:", len(rag_docs))
print("Example doc:")
print(rag_docs[0] if rag_docs else None)

In [ ]:
# Cell 11 - BM25, optional dense retriever, and hybrid retriever
class BM25Retriever:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = docs
        self.k1 = k1
        self.b = b
        self.tokenized_docs = [rag_word_tokenize(doc["text"]) for doc in docs]
        self.doc_lens = [len(tokens) for tokens in self.tokenized_docs]
        self.avgdl = sum(self.doc_lens) / max(1, len(self.doc_lens))
        self.term_freqs = [Counter(tokens) for tokens in self.tokenized_docs]
        df = Counter()
        for tokens in self.tokenized_docs:
            df.update(set(tokens))
        n_docs = len(docs)
        self.idf = {term: max(0.0, math.log(1.0 + (n_docs - freq + 0.5) / (freq + 0.5))) for term, freq in df.items()}

    def search(self, query, top_k=3):
        query_terms = rag_word_tokenize(query)
        scores = []
        for idx, tf in enumerate(self.term_freqs):
            dl = max(1, self.doc_lens[idx])
            score = 0.0
            for term in query_terms:
                freq = tf.get(term, 0)
                if freq == 0:
                    continue
                denom = freq + self.k1 * (1.0 - self.b + self.b * dl / self.avgdl)
                score += self.idf.get(term, 0.0) * (freq * (self.k1 + 1.0)) / denom
            if score > 0:
                scores.append((score, idx))
        best = heapq.nlargest(top_k, scores)
        results = []
        for score, idx in best:
            item = dict(self.docs[idx])
            item["score"] = float(score)
            item["retriever"] = "bm25"
            results.append(item)
        return results


class DenseRetriever:
    def __init__(self, docs, model_name=DENSE_EMBEDDING_MODEL, batch_size=64):
        from sentence_transformers import SentenceTransformer
        self.docs = docs
        self.encoder = SentenceTransformer(model_name, device="cpu")
        texts = [d["text"] for d in docs]
        self.embeddings = self.encoder.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype("float32")

    def search(self, query, top_k=3):
        q = self.encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")[0]
        scores = self.embeddings @ q
        if len(scores) == 0:
            return []
        idxs = np.argpartition(-scores, min(top_k, len(scores)-1))[:top_k]
        idxs = idxs[np.argsort(-scores[idxs])]
        results = []
        for idx in idxs:
            item = dict(self.docs[int(idx)])
            item["score"] = float(scores[int(idx)])
            item["retriever"] = "dense"
            results.append(item)
        return results


class HybridRetriever:
    def __init__(self, bm25_retriever, dense_retriever, rrf_k=60):
        self.bm25 = bm25_retriever
        self.dense = dense_retriever
        self.rrf_k = rrf_k
        self.docs_by_id = {d["doc_id"]: d for d in bm25_retriever.docs}

    def search(self, query, top_k=3):
        bm25_hits = self.bm25.search(query, top_k=max(50, top_k))
        dense_hits = self.dense.search(query, top_k=max(50, top_k))
        scores = defaultdict(float)
        for rank, hit in enumerate(bm25_hits, start=1):
            scores[hit["doc_id"]] += 1.0 / (self.rrf_k + rank)
        for rank, hit in enumerate(dense_hits, start=1):
            scores[hit["doc_id"]] += 1.0 / (self.rrf_k + rank)
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        results = []
        for doc_id, score in ranked:
            item = dict(self.docs_by_id[doc_id])
            item["score"] = float(score)
            item["retriever"] = "hybrid"
            results.append(item)
        return results


retrievers = {}
retrievers["bm25"] = BM25Retriever(rag_docs)
print("BM25 ready")

if ENABLE_DENSE_RETRIEVAL:
    retrievers["dense"] = DenseRetriever(rag_docs)
    retrievers["hybrid"] = HybridRetriever(retrievers["bm25"], retrievers["dense"])
    print("Dense and hybrid retrievers ready")
else:
    print("Dense/hybrid disabled. Set ENABLE_DENSE_RETRIEVAL=True to run them.")

if ACTIVE_RAG_RETRIEVER not in retrievers:
    print(f"ACTIVE_RAG_RETRIEVER={ACTIVE_RAG_RETRIEVER} not available. Falling back to bm25.")
    ACTIVE_RAG_RETRIEVER = "bm25"

print("Available retrievers:", list(retrievers.keys()))
print("Smoke test:")
for hit in retrievers[ACTIVE_RAG_RETRIEVER].search("what is football", top_k=3):
    print(hit["retriever"], hit["doc_id"], round(hit["score"], 4), hit["text"][:180])

In [ ]:
# Cell 12 - Load one PEFT model for all four generation modes
# Base Qwen and RAG-only are produced by temporarily disabling the LoRA adapter.

if "trainer" in globals():
    del trainer
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

assert ADAPTER_DIR.exists(), f"LoRA adapter not found: {ADAPTER_DIR}. Train first or set ADAPTER_DIR to an existing adapter."

infer_tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), use_fast=True)
if infer_tokenizer.pad_token_id is None:
    infer_tokenizer.pad_token = infer_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
).cuda()

chat_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
chat_model.eval()
chat_model.generation_config.pad_token_id = infer_tokenizer.pad_token_id
chat_model.generation_config.eos_token_id = infer_tokenizer.eos_token_id

print("Inference model loaded.")

In [ ]:
# Cell 13 - Prompt builders and four-system generation
RAG_SYSTEM_PROMPT = (
    "You are a helpful conversational assistant. Use the retrieved evidence when it is relevant. "
    "Do not invent unsupported factual claims. If the evidence is insufficient, give a cautious answer."
)


def build_plain_prompt(history, user_text):
    history = [] if history is None else history
    history_for_prompt = history[-MAX_HISTORY_TURNS:] + [{"role": "user", "content": normalize_text(user_text)}]
    return render_chatml(history_for_prompt, add_generation_prompt=True, system_prompt=SYSTEM_PROMPT)


def format_evidence(evidence_list):
    if not evidence_list:
        return "No relevant evidence was retrieved."
    blocks = []
    for i, ev in enumerate(evidence_list, start=1):
        blocks.append(
            f"[{i}] title={ev.get('title', '')} doc_id={ev.get('doc_id', '')} score={ev.get('score', 0.0):.4f}\n{ev.get('text', '')}"
        )
    return "\n\n".join(blocks)


def build_rag_prompt(history, user_text, evidence_list):
    history = [] if history is None else history
    parts = [f"<|im_start|>system\n{RAG_SYSTEM_PROMPT}<|im_end|>\n"]
    for msg in history[-MAX_HISTORY_TURNS:]:
        parts.append(f"<|im_start|>{msg['role']}\n{normalize_text(msg['content'])}<|im_end|>\n")
    parts.append(
        f"<|im_start|>user\nQuestion:\n{normalize_text(user_text)}\n\nRetrieved evidence:\n{format_evidence(evidence_list)}<|im_end|>\n"
    )
    parts.append("<|im_start|>assistant\n")
    return "".join(parts)


def make_retrieval_query(history, user_text):
    history = [] if history is None else history
    previous_user_turns = [m["content"] for m in history[-MAX_HISTORY_TURNS:] if m.get("role") == "user"]
    return " ".join(normalize_text(x) for x in (previous_user_turns[-2:] + [user_text]))


def get_adapter_context(use_lora):
    if use_lora:
        return nullcontext()
    if hasattr(chat_model, "disable_adapter"):
        return chat_model.disable_adapter()
    raise RuntimeError("This PEFT version does not expose disable_adapter().")


@torch.inference_mode()
def generate_from_prompt(prompt_text, use_lora=True, max_new_tokens=MAX_NEW_TOKENS):
    encoded = infer_tokenizer(prompt_text, add_special_tokens=False, return_tensors="pt")
    input_ids = encoded["input_ids"]
    attention_mask = torch.ones_like(input_ids)

    if input_ids.shape[1] > MAX_INFER_TOKENS:
        input_ids = input_ids[:, -MAX_INFER_TOKENS:]
        attention_mask = attention_mask[:, -MAX_INFER_TOKENS:]

    input_ids = input_ids.cuda()
    attention_mask = attention_mask.cuda()

    start = time.perf_counter()
    with get_adapter_context(use_lora):
        output = chat_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            use_cache=True,
            pad_token_id=infer_tokenizer.pad_token_id,
            eos_token_id=infer_tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - start

    new_tokens = output[0, input_ids.shape[1]:]
    reply = infer_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    reply = reply.replace("<|im_end|>", "").strip()
    return reply, latency


def run_system(system_name, user_text, history=None, retriever_name=ACTIVE_RAG_RETRIEVER, top_k=RETRIEVAL_TOP_K, max_new_tokens=MAX_NEW_TOKENS):
    history = [] if history is None else history
    system_name = system_name.lower().strip()
    evidence = []

    if system_name == "base_qwen":
        prompt = build_plain_prompt(history, user_text)
        response, latency = generate_from_prompt(prompt, use_lora=False, max_new_tokens=max_new_tokens)

    elif system_name == "lora_only":
        prompt = build_plain_prompt(history, user_text)
        response, latency = generate_from_prompt(prompt, use_lora=True, max_new_tokens=max_new_tokens)

    elif system_name == "rag_only":
        query = make_retrieval_query(history, user_text)
        evidence = retrievers[retriever_name].search(query, top_k=top_k)
        prompt = build_rag_prompt(history, user_text, evidence)
        response, latency = generate_from_prompt(prompt, use_lora=False, max_new_tokens=max_new_tokens)

    elif system_name == "lora_rag":
        query = make_retrieval_query(history, user_text)
        evidence = retrievers[retriever_name].search(query, top_k=top_k)
        prompt = build_rag_prompt(history, user_text, evidence)
        response, latency = generate_from_prompt(prompt, use_lora=True, max_new_tokens=max_new_tokens)

    else:
        raise ValueError("system_name must be one of: base_qwen, lora_only, rag_only, lora_rag")

    return {"system": system_name, "response": response, "evidence": evidence, "latency_seconds": latency}


# Smoke test all four systems.
for system_name in ["base_qwen", "lora_only", "rag_only", "lora_rag"]:
    result = run_system(system_name, "What is football?", history=[])
    print("\n" + "=" * 80)
    print("SYSTEM:", system_name)
    print("LATENCY:", round(result["latency_seconds"], 3), "s")
    if result["evidence"]:
        print("EVIDENCE:")
        for ev in result["evidence"]:
            print("-", ev["doc_id"], "score=", round(ev["score"], 4), ev["text"][:160])
    print("RESPONSE:")
    print(result["response"])

In [ ]:
# Cell 14 - Interactive chat loop with selectable mode
CHAT_MODE = "lora_rag"   # base_qwen, lora_only, rag_only, lora_rag
history = []

print("Commands: /mode base_qwen | /mode lora_only | /mode rag_only | /mode lora_rag | /reset | quit")

while True:
    user_text = input("User: ").strip()
    if not user_text:
        continue
    if user_text.lower() in {"exit", "quit"}:
        break
    if user_text.lower() == "/reset":
        history = []
        print("History cleared.\n")
        continue
    if user_text.lower().startswith("/mode "):
        mode = user_text.split("/mode ", 1)[1].strip().lower()
        if mode in {"base_qwen", "lora_only", "rag_only", "lora_rag"}:
            CHAT_MODE = mode
            print("Mode changed to:", CHAT_MODE, "\n")
        else:
            print("Invalid mode.\n")
        continue

    result = run_system(CHAT_MODE, user_text, history=history)
    reply = result["response"]
    print(f"\nAssistant [{CHAT_MODE}]: {reply}\n")

    if result["evidence"]:
        print("Retrieved evidence:")
        for i, ev in enumerate(result["evidence"], start=1):
            print(f"[{i}] {ev['doc_id']} | score={ev['score']:.4f}")
            print(ev["text"][:300])
            print()

    history = history + [{"role": "user", "content": user_text}, {"role": "assistant", "content": reply}]
    history = history[-MAX_HISTORY_TURNS:]

In [ ]:
# Cell 15 - Build held-out dialogue contexts for four-system evaluation

def build_heldout_contexts(json_paths, max_examples=50, seed=SEED):
    contexts = []
    for json_path in json_paths:
        conversations = load_conversations(json_path)
        for conv_idx, conv in enumerate(conversations):
            history = []
            for turn_idx, turn in enumerate(conv.get("content", [])):
                text = normalize_text(turn.get("message", ""))
                if not text:
                    continue
                role = "assistant" if turn.get("agent") == "agent_2" else "user"
                if role == "assistant" and history and history[-1]["role"] == "user":
                    contexts.append({
                        "example_id": f"{Path(json_path).stem}_{conv_idx}_{turn_idx}",
                        "history": history[:-1][-MAX_HISTORY_TURNS:],
                        "user_text": history[-1]["content"],
                        "reference_answer": text,
                    })
                history.append({"role": role, "content": text})
                history = history[-MAX_HISTORY_TURNS:]
    rng = random.Random(seed)
    rng.shuffle(contexts)
    return contexts[:max_examples]

heldout_contexts = build_heldout_contexts([VALID_FREQ_PATH, VALID_RARE_PATH], max_examples=N_EVAL_CONTEXTS)
print("heldout contexts:", len(heldout_contexts))
print(heldout_contexts[0])

In [ ]:
# Cell 16 - Automatic text metrics and four-system generation table
SYSTEMS = ["base_qwen", "lora_only", "rag_only", "lora_rag"]


def metric_tokens(text):
    return re.findall(r"[a-zA-Z0-9]+", str(text).lower())


def token_f1(pred, ref):
    pred_tokens = metric_tokens(pred)
    ref_tokens = metric_tokens(ref)
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)
    overlap = sum((pred_counts & ref_counts).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def lcs_len(a, b):
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        curr = [0]
        for j, y in enumerate(b, start=1):
            curr.append(prev[j-1] + 1 if x == y else max(prev[j], curr[-1]))
        prev = curr
    return prev[-1]


def rouge_l_f1(pred, ref):
    pred_tokens = metric_tokens(pred)
    ref_tokens = metric_tokens(ref)
    if not pred_tokens or not ref_tokens:
        return 0.0
    lcs = lcs_len(pred_tokens, ref_tokens)
    if lcs == 0:
        return 0.0
    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def evidence_overlap(response, evidence):
    if not evidence:
        return np.nan
    response_tokens = set(metric_tokens(response))
    evidence_tokens = set(metric_tokens(" ".join(ev.get("text", "") for ev in evidence)))
    response_tokens = {t for t in response_tokens if len(t) > 2}
    if not response_tokens:
        return np.nan
    return len(response_tokens & evidence_tokens) / len(response_tokens)


def evidence_to_text(evidence):
    blocks = []
    for i, ev in enumerate(evidence, start=1):
        blocks.append(
            f"[{i}] retriever={ev.get('retriever', '')}; doc_id={ev.get('doc_id', '')}; title={ev.get('title', '')}; score={ev.get('score', 0.0):.4f}; text={ev.get('text', '')}"
        )
    return "\n\n".join(blocks)


rows = []
for ex_idx, ex in enumerate(heldout_contexts, start=1):
    print(f"Example {ex_idx}/{len(heldout_contexts)}")
    for system_name in SYSTEMS:
        result = run_system(system_name, ex["user_text"], history=ex["history"], top_k=RETRIEVAL_TOP_K, max_new_tokens=MAX_NEW_TOKENS)
        rows.append({
            "example_id": ex["example_id"],
            "system": system_name,
            "dialogue_history": "\n".join([f"{m['role']}: {m['content']}" for m in ex["history"]]),
            "user_text": ex["user_text"],
            "reference_answer": ex["reference_answer"],
            "retriever": ACTIVE_RAG_RETRIEVER if system_name in {"rag_only", "lora_rag"} else "none",
            "retrieved_evidence": evidence_to_text(result["evidence"]),
            "generated_response": result["response"],
            "latency_seconds": result["latency_seconds"],
            "token_f1": token_f1(result["response"], ex["reference_answer"]),
            "rouge_l_f1": rouge_l_f1(result["response"], ex["reference_answer"]),
            "evidence_overlap_proxy": evidence_overlap(result["response"], result["evidence"]),
            "response_length_tokens": len(metric_tokens(result["response"])),
            # Fill manually for final paper-quality human evaluation.
            "relevance_1_5": "",
            "coherence_1_5": "",
            "topic_continuity_1_5": "",
            "factuality_1_5": "",
            "grounding_1_5": "",
            "helpfulness_1_5": "",
            "hallucination_0_1": "",
            "notes": "",
        })

comparison_df = pd.DataFrame(rows)
comparison_path = "/kaggle/working/four_system_outputs_for_eval.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)
display(comparison_df.head())

In [ ]:
# Cell 17 - Automatic comparison summary for paper draft
summary_df = (
    comparison_df
    .groupby("system")
    .agg(
        mean_token_f1=("token_f1", "mean"),
        mean_rouge_l_f1=("rouge_l_f1", "mean"),
        mean_evidence_overlap_proxy=("evidence_overlap_proxy", "mean"),
        mean_latency_seconds=("latency_seconds", "mean"),
        p95_latency_seconds=("latency_seconds", lambda x: float(np.percentile(x, 95))),
        mean_response_length_tokens=("response_length_tokens", "mean"),
    )
    .reset_index()
)
summary_path = "/kaggle/working/four_system_automatic_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)
display(summary_df)

In [ ]:
# Cell 17.5 - Conditional perplexity for Base, LoRA, RAG, and LoRA+RAG

PPL_MAX_CONTEXTS = min(50, len(heldout_contexts))
PPL_MAX_LENGTH = 768

def build_ppl_prompt_and_target(system_name, ex, retriever_name=ACTIVE_RAG_RETRIEVER, top_k=RETRIEVAL_TOP_K):
    system_name = system_name.lower().strip()

    history = ex["history"]
    user_text = ex["user_text"]
    reference_answer = normalize_text(ex["reference_answer"])

    evidence = []

    if system_name in {"base_qwen", "lora_only"}:
        prompt_text = build_plain_prompt(history, user_text)

    elif system_name in {"rag_only", "lora_rag"}:
        query = make_retrieval_query(history, user_text)
        evidence = retrievers[retriever_name].search(query, top_k=top_k)
        prompt_text = build_rag_prompt(history, user_text, evidence)

    else:
        raise ValueError("Invalid system name.")

    target_text = reference_answer + "<|im_end|>\n"

    use_lora = system_name in {"lora_only", "lora_rag"}

    return prompt_text, target_text, use_lora


@torch.no_grad()
def conditional_nll_for_example(system_name, ex):
    prompt_text, target_text, use_lora = build_ppl_prompt_and_target(system_name, ex)

    prompt_ids = infer_tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    target_ids = infer_tokenizer(
        target_text,
        add_special_tokens=False,
    )["input_ids"]

    if len(target_ids) < 1:
        return None

    total_len = len(prompt_ids) + len(target_ids)

    if total_len > PPL_MAX_LENGTH:
        overflow = total_len - PPL_MAX_LENGTH

        # Skip examples where target itself is too long.
        if overflow >= len(prompt_ids):
            return None

        prompt_ids = prompt_ids[overflow:]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)

    batch = {
        "input_ids": torch.tensor([input_ids], dtype=torch.long).cuda(),
        "attention_mask": torch.tensor([attention_mask], dtype=torch.long).cuda(),
        "labels": torch.tensor([labels], dtype=torch.long).cuda(),
    }

    with get_adapter_context(use_lora):
        outputs = chat_model(**batch)
        loss = outputs.loss

    if not torch.isfinite(loss):
        return None

    target_token_count = len(target_ids)
    nll_sum = float(loss.item()) * target_token_count

    return nll_sum, target_token_count


def compute_system_perplexity(system_name, contexts):
    total_nll = 0.0
    total_tokens = 0
    used_examples = 0
    skipped_examples = 0

    chat_model.eval()

    for ex in contexts:
        result = conditional_nll_for_example(system_name, ex)

        if result is None:
            skipped_examples += 1
            continue

        nll_sum, token_count = result
        total_nll += nll_sum
        total_tokens += token_count
        used_examples += 1

    if total_tokens == 0:
        return {
            "system": system_name,
            "eval_loss": float("nan"),
            "perplexity": float("nan"),
            "used_examples": used_examples,
            "skipped_examples": skipped_examples,
            "target_tokens": total_tokens,
        }

    eval_loss = total_nll / total_tokens
    perplexity = math.exp(eval_loss) if math.isfinite(eval_loss) and eval_loss < 20 else float("inf")

    return {
        "system": system_name,
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "used_examples": used_examples,
        "skipped_examples": skipped_examples,
        "target_tokens": total_tokens,
    }


ppl_contexts = heldout_contexts[:PPL_MAX_CONTEXTS]

ppl_rows = []
for system_name in ["base_qwen", "lora_only", "rag_only", "lora_rag"]:
    print("Computing perplexity for:", system_name)
    row = compute_system_perplexity(system_name, ppl_contexts)
    ppl_rows.append(row)

perplexity_df = pd.DataFrame(ppl_rows)

perplexity_path = "/kaggle/working/four_system_perplexity_summary.csv"
perplexity_df.to_csv(perplexity_path, index=False)

print("Saved:", perplexity_path)
display(perplexity_df)

In [ ]:
# Cell 18 - Human evaluation summary after you fill four_system_outputs_for_eval.csv
# Fill the blank columns in /kaggle/working/four_system_outputs_for_eval.csv and upload/run this cell again.

HUMAN_EVAL_CSV = "/kaggle/working/four_system_outputs_for_eval.csv"

human_df = pd.read_csv(HUMAN_EVAL_CSV)
score_cols = [
    "relevance_1_5",
    "coherence_1_5",
    "topic_continuity_1_5",
    "factuality_1_5",
    "grounding_1_5",
    "helpfulness_1_5",
    "hallucination_0_1",
]

for col in score_cols:
    human_df[col] = pd.to_numeric(human_df[col], errors="coerce")

if human_df[score_cols].notna().any().any():
    human_summary = (
        human_df
        .groupby("system")
        .agg(
            relevance=("relevance_1_5", "mean"),
            coherence=("coherence_1_5", "mean"),
            topic_continuity=("topic_continuity_1_5", "mean"),
            factuality=("factuality_1_5", "mean"),
            grounding=("grounding_1_5", "mean"),
            helpfulness=("helpfulness_1_5", "mean"),
            hallucination_rate=("hallucination_0_1", "mean"),
            p95_latency=("latency_seconds", lambda x: float(np.percentile(x, 95))),
        )
        .reset_index()
    )
    human_summary_path = "/kaggle/working/four_system_human_summary.csv"
    human_summary.to_csv(human_summary_path, index=False)
    print("Saved:", human_summary_path)
    display(human_summary)
else:
    print("Manual score columns are still blank. Fill the CSV, then rerun this cell.")

In [ ]:
# Cell 19 - Create retrieval judgment sheet for Recall@k, MRR, nDCG@5
# Manually mark rel_1..rel_5 as 0 or 1 for each retrieved document, then run Cell 20.

retrieval_rows = []
for ex in heldout_contexts:
    query = make_retrieval_query(ex["history"], ex["user_text"])
    for retriever_name, retriever in retrievers.items():
        hits = retriever.search(query, top_k=RETRIEVAL_EVAL_TOP_K)
        row = {
            "example_id": ex["example_id"],
            "retriever": retriever_name,
            "query": query,
            "user_text": ex["user_text"],
            "reference_answer": ex["reference_answer"],
        }
        for i in range(RETRIEVAL_EVAL_TOP_K):
            if i < len(hits):
                row[f"doc_{i+1}_id"] = hits[i].get("doc_id", "")
                row[f"doc_{i+1}_score"] = hits[i].get("score", "")
                row[f"doc_{i+1}_text"] = hits[i].get("text", "")
            else:
                row[f"doc_{i+1}_id"] = ""
                row[f"doc_{i+1}_score"] = ""
                row[f"doc_{i+1}_text"] = ""
            row[f"rel_{i+1}"] = ""
        retrieval_rows.append(row)

retrieval_judgment_df = pd.DataFrame(retrieval_rows)
retrieval_judgment_path = "/kaggle/working/retrieval_judgment_sheet.csv"
retrieval_judgment_df.to_csv(retrieval_judgment_path, index=False)
print("Saved:", retrieval_judgment_path)
display(retrieval_judgment_df.head())

In [ ]:
# Cell 20 - Compute retrieval metrics from manually filled retrieval_judgment_sheet.csv
RETRIEVAL_JUDGMENT_CSV = "/kaggle/working/retrieval_judgment_sheet.csv"
judgments = pd.read_csv(RETRIEVAL_JUDGMENT_CSV)

for i in range(1, RETRIEVAL_EVAL_TOP_K + 1):
    judgments[f"rel_{i}"] = pd.to_numeric(judgments[f"rel_{i}"], errors="coerce")

rel_cols = [f"rel_{i}" for i in range(1, RETRIEVAL_EVAL_TOP_K + 1)]

if judgments[rel_cols].notna().any().any():
    def row_recall_at_k(row, k):
        vals = [row[f"rel_{i}"] for i in range(1, k + 1)]
        vals = [0 if pd.isna(v) else int(v) for v in vals]
        return int(any(v == 1 for v in vals))

    def row_mrr(row):
        for i in range(1, RETRIEVAL_EVAL_TOP_K + 1):
            v = row[f"rel_{i}"]
            if not pd.isna(v) and int(v) == 1:
                return 1.0 / i
        return 0.0

    def row_ndcg_at_5(row):
        rels = []
        for i in range(1, min(5, RETRIEVAL_EVAL_TOP_K) + 1):
            v = row[f"rel_{i}"]
            rels.append(0 if pd.isna(v) else int(v))
        dcg = sum(rel / math.log2(rank + 1) for rank, rel in enumerate(rels, start=1))
        ideal = sorted(rels, reverse=True)
        idcg = sum(rel / math.log2(rank + 1) for rank, rel in enumerate(ideal, start=1))
        return dcg / idcg if idcg > 0 else 0.0

    judgments["Recall@1"] = judgments.apply(lambda r: row_recall_at_k(r, 1), axis=1)
    judgments["Recall@3"] = judgments.apply(lambda r: row_recall_at_k(r, min(3, RETRIEVAL_EVAL_TOP_K)), axis=1)
    judgments["Recall@5"] = judgments.apply(lambda r: row_recall_at_k(r, min(5, RETRIEVAL_EVAL_TOP_K)), axis=1)
    judgments["MRR"] = judgments.apply(row_mrr, axis=1)
    judgments["nDCG@5"] = judgments.apply(row_ndcg_at_5, axis=1)

    retrieval_metrics = judgments.groupby("retriever")[["Recall@1", "Recall@3", "Recall@5", "MRR", "nDCG@5"]].mean().reset_index()
    retrieval_metrics_path = "/kaggle/working/retrieval_metrics_summary.csv"
    retrieval_metrics.to_csv(retrieval_metrics_path, index=False)
    print("Saved:", retrieval_metrics_path)
    display(retrieval_metrics)
else:
    print("rel_1..rel_5 are still blank. Fill the retrieval judgment CSV, then rerun this cell.")

In [ ]:
# Cell 21 - Side-by-side example table for paper figures
# Pick one example and show prompt, evidence, and generated response side by side.
example = heldout_contexts[0]
example_rows = []

for system_name in SYSTEMS:
    result = run_system(system_name, example["user_text"], history=example["history"], top_k=RETRIEVAL_TOP_K)
    example_rows.append({
        "system": system_name,
        "prompt_user_turn": example["user_text"],
        "retrieved_evidence": evidence_to_text(result["evidence"]),
        "generated_response": result["response"],
    })

example_table = pd.DataFrame(example_rows)
example_table_path = "/kaggle/working/side_by_side_example_for_paper.csv"
example_table.to_csv(example_table_path, index=False)
print("Saved:", example_table_path)
display(example_table)